<a href="https://colab.research.google.com/github/Ruba-Alharbi/MentalQA_NER-Reproduction/blob/main/RP__Kappa_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.metrics import cohen_kappa_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd

file_path = 'Entities&Relations_MentalQA_500_data.xlsx'
sheet_name = 'Kappa_drug'

df = pd.read_excel(file_path, sheet_name=sheet_name)


display(df.head())

,Tweet ID,Medication Mentioned 1,Medication Mentioned 2,Question
0,1,سبرالكس,سبرالكس,كنت اعانى من قلق واخدت دواء سبرالكس لمدة 4 شهو...
1,2,"medizapin ,S-citap ,Avlocardyl","medizapin, s-citap, AVLOCARDYL",عندي رهاب اجتماعي احس بديق النفس استعمل mediza...
2,3,"Aripiprazole ,Esciatalopram ,السركويل","Aripiprazole, esciatalopram, السركويل",السلام عليكم سبق وأن أستفسرت وأفادني أحد الأطب...
3,4,لوسترال,لوسترال,"اعاني من القلق , وصفلي الدكتور لوسترال ٥٠ ملج ..."
4,5,ميرزاجن,ميرزاجن,هل حبوب ميرزاجن لها اضرار


In [ ]:
from sklearn.metrics import cohen_kappa_score

# Convert columns to string type, handling potential non-string values like floats (NaNs)
df['Medication Mentioned 1'] = df['Medication Mentioned 1'].astype(str)
df['Medication Mentioned 2'] = df['Medication Mentioned 2'].astype(str)


# Calculate Cohen's Kappa score
kappa = cohen_kappa_score(df['Medication Mentioned 1'], df['Medication Mentioned 2'])

# Display the result
print(f"Cohen's Kappa score: {kappa}")

Cohen's Kappa score: 0.4741613780598368


In [ ]:
import re
from sklearn.metrics import cohen_kappa_score

# normalization helpers
def normalize_arabic(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub("[أإآٱ]", "ا", text)
    text = text.replace("ة", "ه").replace("ى", "ي").replace("ؤ", "و").replace("ئ", "ي")
    text = re.sub(r"^(?:ال)+", "", text)  # remove leading "ال"
    return text.strip()

def canonicalize_cell(cell):
    if pd.isna(cell):
        return ""
    s = str(cell)
    s = s.replace("،", ",")
    tokens = [normalize_arabic(tok.strip()) for tok in s.split(",") if tok.strip()]
    tokens = sorted(set(tokens))  # deduplicate + sort (order-insensitive)
    return ",".join(tokens)

# Apply to both columns
df["ann1"] = df["Medication Mentioned 1"].apply(canonicalize_cell)
df["ann2"] = df["Medication Mentioned 2"].apply(canonicalize_cell)

# Compute Cohen's kappa
kappa = cohen_kappa_score(df["ann1"], df["ann2"])
print("Cohen's Kappa (normalized):", kappa)

Cohen's Kappa (normalized): 0.9646125686394142


In [ ]:
# show mismatches for inspection
disagreements = df[df["ann1"] != df["ann2"]][["Medication Mentioned 1","Medication Mentioned 2","ann1","ann2"]]
disagreements

,Medication Mentioned 1,Medication Mentioned 2,ann1,ann2
23,"سيتالوبرام ,ميرتازبين , كيروفيت ,فينبستين ,بير...","سيتالوبرام, ميرتازبين, توبمود, كيروفيت, فينبوس...","بيراسيتام,سيتالوبرام,فينبستين,فينكامين,كيروفيت...","بيراسيتام,توبمود,سيتالوبرام,فينبوستين,فينكامين..."
56,"تجريتول ,ايزوميبرازول","تجريتول, كاربامازيبين, ايزوميبرازول","ايزوميبرازول,تجريتول","ايزوميبرازول,تجريتول,كاربامازيبين"
